In [5]:
import json
import os
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.parse import quote

import numpy as np

os.environ["POLARS_OOC_MEMORY_BUDGET_MB"] = "100"  # disable Polars' own memory budget
os.environ["POLARS_ENGINE_AFFINITY"] = "streaming"
os.environ["POLARS_STREAMING_CHUNK_SIZE"] = "2000"  # small chunks to stress memory
os.environ["POLARS_VERBOSE"] = "0"  # enable verbose logging to stdout for debugging
os.environ["POLARS_MAX_THREADS"] = "1"  # enable verbose logging to stdout for debugging

import polars as pl

_IMPL = Path(".").parent / "_memory_analysis_impl.py"
assert _IMPL.exists(), f"Expected {_IMPL} to exist"


def write_partitioned_measurements(src: Path, n_partitions: int, rows_per_partition: int) -> float:
    """Write partitioned parquet with (measurement, unit, timestamp, value) columns.

    Each partition holds ``ROWS_PER_PARTITION`` readings spread over
    ``N_MEASUREMENTS`` distinct measurements, so grouping by measurement yields
    a handful of groups with very long ``timestamp``/``value`` list columns.
    """
    total_mbytes = 0
    for p in range(n_partitions):
        measurement = f"measurement_{p}"
        measurement_col = "measurement"
        partition_dir = src / f"{quote(measurement_col)}={quote(measurement)}"
        partition_dir.mkdir(parents=True, exist_ok=True)
        df = pl.DataFrame([
            pl.repeat(measurement, 10_000, eager=True).rename(measurement_col),
            (pl.int_range(0, 10_000, eager=True) % 5)
            .cast(pl.String)
            .str.pad_start(length=3, fill_char="0")
            .rename("channel"),
            pl.Series(np.random.rand(10_000)).rename("value"),
        ])
        df.write_parquet(partition_dir / "00001.parquet")
        total_mbytes += df.estimated_size(unit="mb")
    return total_mbytes


def _run(name: str, tmp_path: Path, n_partitions: int, rows_per_partition: int) -> dict:
    """Spawn a fresh interpreter to run ``run_<name>`` in ``_memory_analysis_impl.py``."""
    src = tmp_path / "src"
    src.mkdir(parents=True, exist_ok=True)
    data_size_mb = write_partitioned_measurements(src, n_partitions, rows_per_partition)

    result = subprocess.run(
        [
            sys.executable,
            str(_IMPL),
            name,
            str(tmp_path),
            str(rows_per_partition),
        ],
        capture_output=True,
        text=True,
        env={**os.environ},
    )
    output = (result.stdout + result.stderr).strip()
    print(f"Output from memory worker '{name}':\n{output}")

    if result.returncode != 0:
        raise RuntimeError(f"Memory worker '{name}' failed:\n{output}")
    else:
        data = json.loads(output.splitlines()[-1])
        data.update({"dataset_size_mb": data_size_mb})
        return data


rows_per_partition = 100_000
num_partitions = 200

results = []

for i in range(1, num_partitions + 1, 20):
    with TemporaryDirectory() as tmp_dir:
        tmp_path = Path(tmp_dir)
        result = _run("pure_polars", tmp_path, i, rows_per_partition)
        print(
            f"Peak RSS: {result['peak_rss_mb']:.1f} MB, Dataset size: {result['dataset_size_mb']:.1f} MB"
        )

        result.update({"n_partitions": i, "rows_per_partition": rows_per_partition})
        results.append(result)

results_df = pl.DataFrame(results)
results_df.unpivot(
    index="n_partitions",
    on=["peak_rss_mb", "dataset_size_mb"],
    variable_name="metric",
    value_name="value",
).plot.line().encode(x="n_partitions", y="value", color="metric")

Output from memory worker 'pure_polars':
{"peak_rss_mb": 0.0}
Peak RSS: 0.0 MB, Dataset size: 0.2 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 17.171875}
Peak RSS: 17.2 MB, Dataset size: 4.9 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 22.3125}
Peak RSS: 22.3 MB, Dataset size: 9.7 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 27.203125}
Peak RSS: 27.2 MB, Dataset size: 14.4 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 32.78125}
Peak RSS: 32.8 MB, Dataset size: 19.2 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 37.59375}
Peak RSS: 37.6 MB, Dataset size: 24.0 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 42.1875}
Peak RSS: 42.2 MB, Dataset size: 29.0 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 47.453125}
Peak RSS: 47.5 MB, Dataset size: 33.9 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 52.203125}
Peak RSS: 52.2 MB, Dataset size: 38.9 MB
Output from memory worker 'pure_pol

alt.Chart(...)

In [6]:
rows_per_partition = 100_000
num_partitions = 200

results = []

for i in range(1, num_partitions + 1, 20):
    with TemporaryDirectory() as tmp_dir:
        tmp_path = Path(tmp_dir)
        result = _run("by_partition", tmp_path, i, rows_per_partition)
        print(
            f"Peak RSS: {result['peak_rss_mb']:.1f} MB, Dataset size: {result['dataset_size_mb']:.1f} MB"
        )

        result.update({"n_partitions": i, "rows_per_partition": rows_per_partition})
        results.append(result)

results_df = pl.DataFrame(results)
results_df.unpivot(
    index="n_partitions",
    on=["peak_rss_mb", "dataset_size_mb"],
    variable_name="metric",
    value_name="value",
).plot.line().encode(x="n_partitions", y="value", color="metric")

Output from memory worker 'by_partition':
{"peak_rss_mb": 62.609375}
Peak RSS: 62.6 MB, Dataset size: 0.2 MB
Output from memory worker 'by_partition':
{"peak_rss_mb": 125.09375}
Peak RSS: 125.1 MB, Dataset size: 4.9 MB
Output from memory worker 'by_partition':
{"peak_rss_mb": 162.046875}
Peak RSS: 162.0 MB, Dataset size: 9.7 MB
Output from memory worker 'by_partition':
{"peak_rss_mb": 190.921875}
Peak RSS: 190.9 MB, Dataset size: 14.4 MB
Output from memory worker 'by_partition':
{"peak_rss_mb": 208.671875}
Peak RSS: 208.7 MB, Dataset size: 19.2 MB
Output from memory worker 'by_partition':
{"peak_rss_mb": 227.234375}
Peak RSS: 227.2 MB, Dataset size: 24.0 MB
Output from memory worker 'by_partition':
{"peak_rss_mb": 227.90625}
Peak RSS: 227.9 MB, Dataset size: 29.0 MB
Output from memory worker 'by_partition':
{"peak_rss_mb": 225.875}
Peak RSS: 225.9 MB, Dataset size: 33.9 MB
Output from memory worker 'by_partition':
Traceback (most recent call last):
  File "/Users/nennia/Projects/data-w

RuntimeError: Memory worker 'by_partition' failed:
Traceback (most recent call last):
  File "/Users/nennia/Projects/data-warehousing-with-polars/notebooks/_memory_analysis_impl.py", line 113, in <module>
    globals()[f"run_{_name}"](_tmp, _rows_per_partition)
  File "/Users/nennia/Projects/data-warehousing-with-polars/notebooks/_memory_analysis_impl.py", line 100, in run_by_partition
    pipeline.run()
  File "/Users/nennia/Projects/data-warehousing-with-polars/packages/data-warehousing-with-polars/src/data_warehousing_with_polars/incremental.py", line 780, in run
    batch = src.poll(cursors.get(i))
            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/nennia/Projects/data-warehousing-with-polars/packages/data-warehousing-with-polars/src/data_warehousing_with_polars/incremental.py", line 349, in poll
    frame = _scan_files(new_files, self._file_format, self._reader_kwargs, self._concat_options)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^